# NHS A&E Waiting Times — Predicting Pressure Points
## Notebook 03: Feature Engineering

**Author:** David  
**Date:** April 2026  
**Stage:** Feature Engineering  

### Objective
Create new variables (features) that give the machine learning model 
better signals to learn from. Raw data alone is not enough — we need 
to engineer features that capture patterns the model cannot see directly.

### Features We Will Create
1. **Previous month breach rate** — strongest predictor of next month
2. **3-month rolling average** — captures momentum and trend
3. **Is winter flag** — captures seasonal pressure (Oct-Feb = 1)
4. **Trust historical average** — how does this Trust normally perform
5. **Month-on-month attendance change** — is the Trust getting busier
6. **Encoded Trust ID** — converts names to numbers for the model

### Why This Matters
Feature engineering is where domain knowledge meets data science. 
Understanding that NHS pressure is seasonal, Trust-specific, and 
momentum-driven allows us to build features that capture exactly 
those patterns — making the model significantly more accurate.

In [1]:
import pandas as pd
import numpy as np
import os

# Load the clean dataset
df = pd.read_csv('../data/processed/nhs_ae_clean.csv', 
                 parse_dates=['Period_date'])

print(f"Dataset loaded: {df.shape}")
print(f"Trusts: {df['Org name'].nunique()}")
print(f"Date range: {df['Period_date'].min().strftime('%B %Y')} "
      f"to {df['Period_date'].max().strftime('%B %Y')}")

Dataset loaded: (4381, 28)
Trusts: 132
Date range: April 2023 to March 2026


In [2]:
# Fix — remove the TOTAL row that slipped through
print(f"Rows before: {len(df)}")
df = df[~df['Org name'].str.upper().str.contains('TOTAL', na=False)]
print(f"Rows after removing TOTAL: {len(df)}")

# Also remove the two merged Trusts with non-comparable attendance figures
merged_trusts = [
    'UNIVERSITY HOSPITALS BIRMINGHAM NHS FOUNDATION TRUST',
    'MID AND SOUTH ESSEX NHS FOUNDATION TRUST'
]

print(f"\nNote: Keeping merged Trusts in dataset for now.")
print(f"They will be flagged as a feature rather than removed.")
print(f"This preserves data while letting the model account for them.")

# Save the corrected clean file — overwrite the previous version
df.to_csv('../data/processed/nhs_ae_clean.csv', index=False)
print(f"\nCorrected dataset saved.")
print(f"Final shape: {df.shape}")

Rows before: 4381
Rows after removing TOTAL: 4380

Note: Keeping merged Trusts in dataset for now.
They will be flagged as a feature rather than removed.
This preserves data while letting the model account for them.

Corrected dataset saved.
Final shape: (4380, 28)


### Data Consistency Fix

When loading the cleaned dataset, 4,381 rows were detected instead of 
the expected 4,380. Investigation revealed that one TOTAL summary row 
was removed from memory during Notebook 02 but was never saved back 
to the CSV file on disk.

This has been corrected here and the clean CSV overwritten with the 
accurate 4,380 row version. This ensures anyone who clones this 
repository and runs the notebooks in order will get consistent results 
across all notebooks.

In [3]:
# Sort data correctly before creating lag features
# Critical — lag features only work if data is sorted by Trust and date
df = df.sort_values(['Org name', 'Period_date']).reset_index(drop=True)

print("Data sorted by Trust and date.")
print(f"\nSample — first 4 rows for one Trust:")
sample = df[df['Org name'] == 'HARROGATE AND DISTRICT NHS FOUNDATION TRUST']
print(sample[['Org name', 'Period_date', 'breach_rate']].head(4).to_string(index=False))

Data sorted by Trust and date.

Sample — first 4 rows for one Trust:
                                   Org name Period_date  breach_rate
HARROGATE AND DISTRICT NHS FOUNDATION TRUST  2023-04-01    19.560290
HARROGATE AND DISTRICT NHS FOUNDATION TRUST  2023-05-01    22.158273
HARROGATE AND DISTRICT NHS FOUNDATION TRUST  2023-06-01    23.425795
HARROGATE AND DISTRICT NHS FOUNDATION TRUST  2023-07-01    21.753447


In [4]:
# ── FEATURE ENGINEERING ───────────────────────────────────────────────────

# Feature 1 — Previous month breach rate (lag 1)
df['prev_month_breach'] = df.groupby('Org name')['breach_rate'].shift(1)

# Feature 2 — 3 month rolling average
df['rolling_3m_avg'] = df.groupby('Org name')['breach_rate'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=2).mean()
)

# Feature 3 — Is winter flag (October to February = 1, else 0)
df['is_winter'] = df['month'].isin([10, 11, 12, 1, 2]).astype(int)

# Feature 4 — Trust historical average breach rate
trust_hist_avg = df.groupby('Org name')['breach_rate'].mean()
df['trust_hist_avg'] = df['Org name'].map(trust_hist_avg)

# Feature 5 — Month on month attendance change
df['attendance_change'] = df.groupby('Org name')[
    'A&E attendances Type 1'].pct_change() * 100

# Feature 6 — Encoded Trust ID (converts name to number)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['trust_encoded'] = le.fit_transform(df['Org name'])

print("All 6 features created successfully.")
print(f"\nNew columns added:")
new_cols = ['prev_month_breach', 'rolling_3m_avg', 'is_winter', 
            'trust_hist_avg', 'attendance_change', 'trust_encoded']
for col in new_cols:
    missing = df[col].isnull().sum()
    print(f"  {col}: {missing} missing values")

All 6 features created successfully.

New columns added:
  prev_month_breach: 131 missing values
  rolling_3m_avg: 261 missing values
  is_winter: 0 missing values
  trust_hist_avg: 0 missing values
  attendance_change: 131 missing values
  trust_encoded: 0 missing values


In [5]:
# Sense check — look at one Trust across several months
# We should see lag features shifting correctly
check = df[df['Org name'] == 'HARROGATE AND DISTRICT NHS FOUNDATION TRUST'][
    ['Period_date', 'breach_rate', 'prev_month_breach', 
     'rolling_3m_avg', 'is_winter', 'trust_hist_avg',
     'attendance_change', 'trust_encoded']
].head(8)

print("Feature sense check — Harrogate and District:")
print(check.to_string(index=False))

Feature sense check — Harrogate and District:
Period_date  breach_rate  prev_month_breach  rolling_3m_avg  is_winter  trust_hist_avg  attendance_change  trust_encoded
 2023-04-01    19.560290                NaN             NaN          0       30.263663                NaN             37
 2023-05-01    22.158273          19.560290             NaN          0       30.263663          10.267452             37
 2023-06-01    23.425795          22.158273       20.859282          0       30.263663           1.521069             37
 2023-07-01    21.753447          23.425795       21.714786          0       30.263663          -1.619761             37
 2023-08-01    32.709883          21.753447       22.445838          0       30.263663          -3.169376             37
 2023-09-01    41.478098          32.709883       25.963042          0       30.263663           2.380446             37
 2023-10-01    35.393258          41.478098       31.980476          1       30.263663          -0.228358  

In [6]:
# Save the feature engineered dataset
output_path = '../data/processed/'
df.to_csv(output_path + 'nhs_ae_features.csv', index=False)

print("=" * 55)
print("NOTEBOOK 03 — FEATURE ENGINEERING SUMMARY")
print("=" * 55)
print(f"Input rows:             4,380")
print(f"Output rows:            {len(df)}")
print(f"Original features:      28")
print(f"New features created:   6")
print(f"Total features:         {len(df.columns)}")
print(f"Features created:")
print(f"  1. prev_month_breach  — lag 1 month breach rate")
print(f"  2. rolling_3m_avg     — 3 month rolling average")
print(f"  3. is_winter          — seasonal flag Oct-Feb")
print(f"  4. trust_hist_avg     — Trust historical average")
print(f"  5. attendance_change  — month on month % change")
print(f"  6. trust_encoded      — numeric Trust identifier")
print(f"Saved to: data/processed/nhs_ae_features.csv")
print("=" * 55)
print("NEXT: Notebook 04 — Machine Learning Model")
print("=" * 55)

NOTEBOOK 03 — FEATURE ENGINEERING SUMMARY
Input rows:             4,380
Output rows:            4380
Original features:      28
New features created:   6
Total features:         34
Features created:
  1. prev_month_breach  — lag 1 month breach rate
  2. rolling_3m_avg     — 3 month rolling average
  3. is_winter          — seasonal flag Oct-Feb
  4. trust_hist_avg     — Trust historical average
  5. attendance_change  — month on month % change
  6. trust_encoded      — numeric Trust identifier
Saved to: data/processed/nhs_ae_features.csv
NEXT: Notebook 04 — Machine Learning Model
